# Práctica



## Preparación previa

### Importaciones

In [1]:
import altair as alt
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import os

### Instalación de Altair

Para la instalación de las librerías necesarias, he creado un entorno en conda llamado `unedviz`, utilizando el comando:

```
conda create --name unedviz python=3.12
conda activate unedviz
```

Después, he procedido a instalar `altair` haciendo:
```
conda install -c conda-forge altair-all
```

Y he creado el kernel para jupyter:
```
python -m ipykernel install --user --name=unedviz --display-name "Python (unedviz)"
```

Finalmente, para comprobar la correcta instalación, he ejecutado el código de muestra de la web de Altair, instalando previamente los datasets de muestra con el comando:
```
pip install vega_datasets
```

In [2]:
# load a sample dataset as a pandas DataFrame
from vega_datasets import data
cars = data.cars()

# make the chart
alt.Chart(cars).mark_point().encode(
    x='Horsepower',
    y='Miles_per_Gallon',
    color='Origin',
).interactive()

alt.Chart(...)

### Carga del dataset

Los datos se encuentran en la carpeta `./CSV Files`. Estos son los archivos que tenemos:

- **`.\CSV Files\The UNSW-NB15 description.pdf`**: Archivo que describe el dataset. Indica que el conjunto de datos UNSW-NB15 fue generado en el laboratorio Cyber Range de UNSW Canberra, combinando tráfico normal real y ataques sintéticos, y contiene más de 2.5 millones de registros con 49 características, distribuidos en archivos CSV y clasificados por tipos de ataques como DoS, Exploits, y Malware. Se utilizaron herramientas como Tcpdump, Argus y Bro-IDS, y se incluyen particiones para entrenamiento (175,341 registros) y prueba (82,332 registros).
- **`.\CSV Files\NUSW-NB15_features.csv`**: Incluye una lista de 49 características, con su nombre, tipo de datos (`integer`, `nominal`, etc.) y descripción.
- **Registros de datos**: contienen los registros verdaderos de datos de tráfico real y ataques sintéticos combinados, y sus etiquetas.
    - **`.\CSV Files\UNSW-NB15_1.csv`**: Primer CSV. Contiene 49 columnas de datos, y cada columna corresponde a una de las características listadas en el archivo `NUSW-NB15_features.csv`. En total contiene 700 001 registros.
    - **`.\CSV Files\UNSW-NB15_2.csv`**: Segundo CSV. La estructura es igual al archivo anterior. En total contiene 700 001 registros.
    - **`.\CSV Files\UNSW-NB15_3.csv`**: Tercer archivo, contiene 700 001 registros.
    - **`.\CSV Files\UNSW-NB15_4.csv`**: Cuarto archivo, contiene 440 044 registros.
- **`.\CSV Files\NUSW-NB15_GT.csv`**: Registra una lista de los eventos de los ataques. Es el *ground truth*, contiene la verdad conocida o etiquetas reales de los datos: es decir, indica con certeza qué tipo específico de ataque es. Actúa como un archivo de referencia independiente, y con una estructura más limpia, usada para validar/relacionar los datos de otra manera. Para cada ataque, incluye su hora de inicio y hora final, la categoría de ataque (p.ej. *Backdoor*, *Exploit*, etc.), subcategoría, protocolo utilizado, IP y puerto origen, IP y puerto destino, nombre del ataque y su referencia (CVE, BID, etc.).
- **`.\CSV Files\UNSW-NB15_LIST_EVENTS.csv`**: Resumen agregado de los eventos. Contiene el número de eventos totales de cada categoría y subcategoría de ataque (datos agregados).
- **`.\CSV Files\Training and Testing Sets\UNSW_NB15_training-set.csv`**: Se trata de una partición de los archivos de datos con 175 341 registros. Sin embargo, el objetivo de este training set en concreto es utilizarlo para el entrenamiento de modelos de Machine Learning.
- **`.\CSV Files\Training and Testing Sets\UNSW_NB15_testing-set.csv`**: Igual que el training set, se trata de una partición de los archivos de datos con 82 332 registros. El objetivo de este testing set es utilizarlo para validar el entrenamiento de modelos de Machine Learning.


In [17]:
# Cargar nombres de columnas
features_path = r'.\CSV Files\NUSW-NB15_features.csv'
features_df = pd.read_csv(features_path, encoding='latin1')
features_df.columns = features_df.columns.str.strip()

# Crear diccionario de conversión de tipos
type_mapping = {
    'Float': 'float32',
    'Integer': 'int32',
    'integer': 'int32',
    'Binary': 'int8',
    'binary': 'int8',
    'nominal': 'category',
    'Timestamp': 'str'
}

# Crear diccionario que asocie cada nombre de la columna (clave) con su tipo de datos (valor)
dtype_dict = {}
for index, row in features_df.iterrows():
    column_name = row['Name']
    column_type = type_mapping.get(row['Type'], 'object')
    dtype_dict[column_name] = column_type


In [32]:
# Archivos de datos sin encabezado
data_files = [
    r'.\CSV Files\UNSW-NB15_1.csv',
    r'.\CSV Files\UNSW-NB15_2.csv',
    r'.\CSV Files\UNSW-NB15_3.csv',
    r'.\CSV Files\UNSW-NB15_4.csv'
]

# Leer y concatenar todos los archivos
dataframes = []
for file in data_files:
    # Leer archivo sin asignar tipos
    df = pd.read_csv(file, header=None, names=list(dtype_dict.keys()), encoding='latin1')
    dataframes.append(df)

# Concatenar los DataFrames
full_data = pd.concat(dataframes, ignore_index=True)

C:\Users\maial\AppData\Local\Temp\ipykernel_75148\1844327936.py:13: DtypeWarning: Columns (1,3,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, header=None, names=list(dtype_dict.keys()), encoding='latin1')
C:\Users\maial\AppData\Local\Temp\ipykernel_75148\1844327936.py:13: DtypeWarning: Columns (3,39,47) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(file, header=None, names=list(dtype_dict.keys()), encoding='latin1')


Carga de los datos realizado con éxito.

## Preparación de los datos (preprocesado)

### Limpieza y asignación de tipos

In [33]:
# Función para intentar convertir un valor en hexadecimal a decimal y '-' a -1
def convert_to_int(value):
    try:
        str_value = str(value).strip()
    
        if str_value.startswith('0x'): # Hexadecimal
            return int(str_value, 16)
        elif str_value in ['', '-']:
            return -1
        else:
            return int(str_value)
    except ValueError:
        return value

In [34]:
# Aplicar la conversión a hexadecimal a decimal en las columnas que deben ser enteros
for column, dtype in dtype_dict.items():
    if dtype == 'int32':
        full_data[column] = full_data[column].apply(convert_to_int).fillna(-2)
    elif dtype == 'int8':
        full_data[column] = full_data[column].fillna(-2)


# Convertir las columnas al tipo correcto según el dtype_dict
for column, dtype in dtype_dict.items():
    full_data[column] = full_data[column].astype(dtype)

srcip-category
sport-int32
dstip-category
dsport-int32
proto-category
state-category
dur-float32
sbytes-int32
dbytes-int32
sttl-int32
dttl-int32
sloss-int32
dloss-int32
service-category
Sload-float32
Dload-float32
Spkts-int32
Dpkts-int32
swin-int32
dwin-int32
stcpb-int32
dtcpb-int32
smeansz-int32
dmeansz-int32
trans_depth-int32
res_bdy_len-int32
Sjit-float32
Djit-float32
Stime-str
Ltime-str
Sintpkt-float32
Dintpkt-float32
tcprtt-float32
synack-float32
ackdat-float32
is_sm_ips_ports-int8
ct_state_ttl-int32
ct_flw_http_mthd-int32
is_ftp_login-int8
ct_ftp_cmd-int32
ct_srv_src-int32
ct_srv_dst-int32
ct_dst_ltm-int32
ct_src_ ltm-int32
ct_src_dport_ltm-int32
ct_dst_sport_ltm-int32
ct_dst_src_ltm-int32
attack_cat-category
Label-int8


In [35]:
print("Mínimo:", full_data['sport'].min())
print("Máximo:", full_data['sport'].max())

Mínimo: -1
Máximo: 65535


In [36]:
print((full_data['sport'] == -1).sum())

2


Hemos hecho estos reemplazos en los valores de los enteros, debido a que las columnas integer no pueden contener valores faltantes:
- fillna(-2)
- value == '-' or ' ': return -1
- Conversion de hexadecimal a decimal.

In [38]:
# Mostrar info general
print(full_data.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2540047 entries, 0 to 2540046
Data columns (total 49 columns):
 #   Column            Dtype   
---  ------            -----   
 0   srcip             category
 1   sport             int32   
 2   dstip             category
 3   dsport            int32   
 4   proto             category
 5   state             category
 6   dur               float32 
 7   sbytes            int32   
 8   dbytes            int32   
 9   sttl              int32   
 10  dttl              int32   
 11  sloss             int32   
 12  dloss             int32   
 13  service           category
 14  Sload             float32 
 15  Dload             float32 
 16  Spkts             int32   
 17  Dpkts             int32   
 18  swin              int32   
 19  dwin              int32   
 20  stcpb             int32   
 21  dtcpb             int32   
 22  smeansz           int32   
 23  dmeansz           int32   
 24  trans_depth       int32   
 25  res_bdy_len       

### Análisis Exploratorio de los Datos (EDA)

In [39]:
# Valores nulos
full_data.isnull().sum()

srcip                     0
sport                     0
dstip                     0
dsport                    0
proto                     0
state                     0
dur                       0
sbytes                    0
dbytes                    0
sttl                      0
dttl                      0
sloss                     0
dloss                     0
service                   0
Sload                     0
Dload                     0
Spkts                     0
Dpkts                     0
swin                      0
dwin                      0
stcpb                     0
dtcpb                     0
smeansz                   0
dmeansz                   0
trans_depth               0
res_bdy_len               0
Sjit                      0
Djit                      0
Stime                     0
Ltime                     0
Sintpkt                   0
Dintpkt                   0
tcprtt                    0
synack                    0
ackdat                    0
is_sm_ips_ports     

In [40]:
full_data.duplicated().sum()

np.int64(480633)

#### Análisis de las Variables Numéricas

In [41]:
# Ver estadística descriptiva
full_data.describe()

,sport,dsport,dur,sbytes,dbytes,sttl,dttl,sloss,dloss,Sload,...,is_ftp_login,ct_ftp_cmd,ct_srv_src,ct_srv_dst,ct_dst_ltm,ct_src_ ltm,ct_src_dport_ltm,ct_dst_sport_ltm,ct_dst_src_ltm,Label
count,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,...,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06,2.540047e+06
mean,3.053447e+04,1.166415e+04,6.587917e-01,4.339600e+03,3.642759e+04,6.278197e+01,3.076681e+01,5.163921e+00,1.632944e+01,3.695645e+07,...,-1.108517e+00,-5.423762e-01,9.206988e+00,8.988958e+00,6.439103e+00,6.900986e+00,4.642139e+00,3.592729e+00,6.845886e+00,1.264870e-01
std,2.044216e+04,4.786167e+05,1.392493e+01,5.640599e+04,1.610960e+05,7.462277e+01,4.285089e+01,2.251707e+01,5.659474e+01,1.186043e+08,...,1.020313e+00,5.506126e-01,1.083676e+01,1.082249e+01,8.162034e+00,8.205062e+00,8.477579e+00,6.174445e+00,1.125828e+01,3.323975e-01
min,-1.000000e+00,-1.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,...,-2.000000e+00,-1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00
25%,1.122600e+04,5.300000e+01,1.037000e-03,2.000000e+02,1.780000e+02,3.100000e+01,2.900000e+01,0.000000e+00,0.000000e+00,1.353963e+05,...,-2.000000e+00,-1.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,2.000000e+00,1.000000e+00,1.000000e+00,1.000000e+00,0.000000e+00
50%,3.168700e+04,8.000000e+01,1.586100e-02,1.470000e+03,1.820000e+03,3.100000e+01,2.900000e+01,3.000000e+00,4.000000e+00,5.893038e+05,...,-2.000000e+00,-1.000000e+00,5.000000e+00,5.000000e+00,3.000000e+00,4.000000e+00,1.000000e+00,1.000000e+00,2.000000e+00,0.000000e+00
75%,4.743900e+04,1.498700e+04,2.145545e-01,3.182000e+03,1.489400e+04,3.100000e+01,2.900000e+01,7.000000e+00,1.400000e+01,2.039923e+06,...,0.000000e+00,0.000000e+00,1.000000e+01,1.000000e+01,6.000000e+00,7.000000e+00,2.000000e+00,1.000000e+00,5.000000e+00,0.000000e+00
max,6.553500e+04,5.389893e+08,8.786638e+03,1.435577e+07,1.465753e+07,2.550000e+02,2.540000e+02,5.319000e+03,5.507000e+03,5.988000e+09,...,4.000000e+00,8.000000e+00,6.700000e+01,6.700000e+01,6.700000e+01,6.700000e+01,6.700000e+01,6.000000e+01,6.700000e+01,1.000000e+00


Ahora veremos la distribución de estas variables numéricas en formato de gráfico, para intentar obtener más detalle:

In [47]:
alt.data_transformers.enable("vegafusion")
# Filtramos solo las columnas numéricas
numeric_cols = full_data.select_dtypes(include=['number'])

# Recorrer cada columna numérica para mostrar distribución
for col in numeric_cols.columns:
    print(f"Distribución de la columna {col}:")
        
    # Histograma
    hist = alt.Chart(full_data).mark_bar().encode(
        alt.X(f'{col}:Q', bin=True),
        alt.Y('count():Q')
    ).properties(title=f'Histograma de {col}')
    
    # Mostrar ambos gráficos
    hist.show()

Distribución de la columna sport:


alt.Chart(...)

Distribución de la columna dsport:


alt.Chart(...)

Distribución de la columna dur:


...

alt.Chart(...)

Distribución de la columna ct_dst_src_ltm:


alt.Chart(...)

Distribución de la columna Label:


alt.Chart(...)

En algunos casos parece que la distribución no tiene sentido, pero lo que realmente pasa es que tenemos outliers que imposibilitan la correcta visualización en el plot.

In [51]:
print(full_data['dsport'].unique())
print(full_data['dsport'].max())
print(full_data['dsport'].min())
print((full_data['dsport'] == -1).sum())

[  53 1024  111 ...  632  186  518]
538989345
-1
7


### Análisis de las Variables Categóricas

In [52]:
# Método para contar y calcular el % de las categorías
def count_and_percent(df, column_name):
    count_data = df[column_name].value_counts().reset_index()
    count_data.columns = [column_name, 'count']
    count_data['%'] = (count_data['count'] / count_data['count'].sum()) * 100
    return count_data

In [55]:
# Aplicar el método a todas las columnas categóricas
categorical_columns = full_data.select_dtypes(include=['category']).columns

# Crear un diccionario o una lista para almacenar los resultados
results = {}

for col in categorical_columns:
    results[col] = count_and_percent(df, col)

# Mostrar resultados (por ejemplo, para la columna 'neighbourhood_group')
for col, result in results.items():
    print(f"Distribución para {col}:")
    print(result)
    print("\n")

Distribución para srcip:
             srcip  count         %
0   149.171.126.14  41037  9.325658
1     175.45.176.1  40538  9.212261
2     175.45.176.0  38614  8.775032
3   149.171.126.10  30356  6.898401
4       59.166.0.1  27391  6.224605
5       59.166.0.4  27216  6.184836
6       59.166.0.5  27164  6.173019
7       59.166.0.0  27111  6.160975
8       59.166.0.2  27050  6.147113
9       59.166.0.3  26900  6.113025
10      59.166.0.9  26398  5.998946
11      59.166.0.7  26055  5.920999
12      59.166.0.8  26024  5.913954
13      59.166.0.6  25590  5.815328
14    175.45.176.2  11805  2.682686
15    175.45.176.3   7981  1.813682
16     10.40.182.6    964  0.219069
17      10.40.85.1    415  0.094309
18     10.40.182.1    403  0.091582
19     10.40.85.10    225  0.051131
20     10.40.85.30    220  0.049995
21     10.40.182.3    219  0.049768
22     10.40.170.2    216  0.049086
23  149.171.126.13    106  0.024089
24  149.171.126.19      7  0.001591
25  149.171.126.18      7  0.001591
26 